In [1]:
#1
import pandas as pd
import re
roll_number = "1024170282"
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]
last_digs =roll_number[-2:]
categories=["billing", "account", "general"]

my_entry=[]

for dig in last_digs:
    d=int(dig)
    category = categories[d%3]

    if d==8:
        my_entry.append({
            "question": "what time does the support service close",
            "answer": "The support service closes at 5 PM.",
            "keywords": "support closing time",
            "category": category
        })
    elif d == 2:
        my_entry.append({
            "question": "where can i find the office location",
            "answer": "You can find the office location on the Contact Us page.",
            "keywords": "office location address",
            "category": category
        })

data = fixed_entries + my_entry
df = pd.DataFrame(data)
print(df)


                                   question  \
0                    what is the annual fee   
1                     how to reset password   
2               what are your working hours   
3                     how can i pay the fee   
4  what time does the support service close   
5      where can i find the office location   

                                              answer                 keywords  \
0                          The annual fee is Rs 500.    fee cost price charge   
1                   Go to Settings > Reset Password.     password reset login   
2                          We are open 9 AM to 5 PM.   hours timing open time   
3         You can pay via UPI, card, or net banking.      pay payment upi fee   
4                The support service closes at 5 PM.     support closing time   
5  You can find the office location on the Contac...  office location address   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  general  


In [4]:
#2
def score_ques(ques,df):
    ques_words=set(re.findall(r'\b\w+\b',ques.lower()))
    results=[]

    for index,row in df.iterrows():
        key_words=set(row["keywords"].lower().split())
        matches = ques_words.intersection(key_words)
        score = len(matches) / len(key_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(matches),
                "confidence": round(score, 2)
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(results)

query = input("\nEnter ques: ")
results = score_ques(query, df)

print("\nQ2: Matching FAQs")
if len(results) == 0:
    print("No matching FAQ found.")
else:
    print(results)
        


Enter ques:  what is the annual fee



Q2: Matching FAQs
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category matched_keywords  confidence  
0  billing              fee        0.25  
1  billing              fee        0.25  


In [7]:
#3
def same_category(cat, df):
    return df[df["category"] == cat]

cat = my_entry[0]["category"]

print("\nQ3: FAQs in", cat, "category:")
print(same_category(cat, df))


Q3: FAQs in general category:
                                   question  \
2               what are your working hours   
4  what time does the support service close   
5      where can i find the office location   

                                              answer                 keywords  \
2                          We are open 9 AM to 5 PM.   hours timing open time   
4                The support service closes at 5 PM.     support closing time   
5  You can find the office location on the Contac...  office location address   

  category  
2  general  
4  general  
5  general  


In [8]:
#4
i = int(input("\nEnter FAQ index: "))
word = input("Enter a new keyword: ")

df.loc[i, "keywords"] = df.loc[i, "keywords"] + " " + word

file = roll_number + "_faq_data.csv"
df.to_csv(file, index=False)

print("Updated FAQ:")
print(df.loc[i])

print("Saved as:", file)


Enter FAQ index:  1
Enter a new keyword:  list


Updated FAQ:
question               how to reset password
answer      Go to Settings > Reset Password.
keywords           password reset login list
category                             account
Name: 1, dtype: object
Saved as: 1024170282_faq_data.csv


In [9]:
#5
count = df.groupby("category").size()
print(count)

category
account    1
billing    2
general    3
dtype: int64


In [ ]:
#6
